# ADPO evaluation on real GitHub Actions data

Reproducible evaluation of ADPO's six deterministic analyzers on CI history from the
public repositories in `repos.yaml`. Every number in `REPORT.md` is computed in this
notebook (the last cell writes it). Regenerate with the command in `research/README.md`.

In [ ]:
import json
import math
import sys
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ruptures as rpt
from scipy import stats

HERE = Path.cwd() if (Path.cwd() / "repos.yaml").exists() else Path.cwd() / "research"
sys.path.insert(0, str(HERE))
from adpo_data import load_tables, to_raw_runs  # noqa: E402

from adpo.analyzers.dependency_install import DependencyInstallAnalyzer, DependencyInstallConfig
from adpo.analyzers.failure_retry import FailureRetryAnalyzer, FailureRetryConfig
from adpo.analyzers.historical_regression import HistoricalRegressionAnalyzer
from adpo.analyzers.parallelization import ParallelizationAnalyzer, ParallelizationConfig
from adpo.analyzers.slow_job import SlowJobAnalyzer, SlowJobConfig
from adpo.analyzers.slow_step import SlowStepAnalyzer, SlowStepConfig
from adpo.engine import AnalysisEngine
from adpo.ingest import parse_workflow_runs

warnings.filterwarnings("ignore", category=RuntimeWarning)
FIG = HERE / "figures"
FIG.mkdir(exist_ok=True)
SEED = 20261112
R = {}  # every number quoted in REPORT.md is stored here
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})


def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG / f"{name}.png")
    plt.close()


tables, SOURCE = load_tables()
meta = json.loads((HERE / "data" / "collection_meta.json").read_text())
REPOS = sorted(tables["runs"].repo.unique())
RUNS = {repo: parse_workflow_runs(to_raw_runs(tables, repo)) for repo in REPOS}
R["source"] = SOURCE
R["collected_at"] = meta["collected_at"]
R["n_repos"] = len(REPOS)
print(SOURCE, len(REPOS), "repos")

## a. Data description

In [ ]:
runs_df = tables["runs"].copy()
jobs_df = tables["jobs"].copy()
steps_df = tables["steps"].copy()
runs_df["base_run"] = runs_df.run_id.str.split("/").str[0]
for df in (jobs_df, steps_df):
    df["started"] = pd.to_datetime(df.started_at, utc=True, errors="coerce")
    df["completed"] = pd.to_datetime(df.completed_at, utc=True, errors="coerce")
    df["duration"] = (df.completed - df.started).dt.total_seconds()
jobs_df = jobs_df.merge(runs_df[["repo", "run_id", "base_run", "created_at", "event", "branch"]], on=["repo", "run_id"])

per_repo = pd.DataFrame(
    {
        "workflow": [meta["repos"][r]["workflow"] for r in REPOS],
        "runs": runs_df.groupby("repo").base_run.nunique(),
        "attempts": runs_df.groupby("repo").size(),
        "jobs": jobs_df.groupby("repo").size(),
        "steps": steps_df.groupby("repo").size(),
        "first_run": runs_df.groupby("repo").created_at.min().str[:10],
        "last_run": runs_df.groupby("repo").created_at.max().str[:10],
        "jobs_matched_to_yaml_pct": (jobs_df.groupby("repo").yaml_key.apply(lambda s: s.notna().mean()) * 100).round(0),
    },
    index=REPOS,
)
R["totals"] = {k: int(per_repo[k].sum()) for k in ("runs", "attempts", "jobs", "steps")}
R["per_repo_table"] = per_repo.to_markdown()
R["yaml_match_pct_overall"] = round(float(jobs_df.yaml_key.notna().mean() * 100), 1)

conc = jobs_df.conclusion.fillna("none").value_counts(normalize=True).mul(100).round(1)
R["job_conclusion_pct"] = conc.to_dict()
run_conc = runs_df[~runs_df.run_id.str.contains("/")].conclusion.fillna("none").value_counts(normalize=True)
R["run_conclusion_pct"] = run_conc.mul(100).round(1).to_dict()
R["event_pct"] = runs_df[~runs_df.run_id.str.contains("/")].event.value_counts(normalize=True).mul(100).round(1).to_dict()

missing_job_timing = jobs_df.duration.isna().mean() * 100
negative = (jobs_df.duration < 0).sum()
step_missing = steps_df.duration.isna().mean() * 100
R["missing"] = {
    "job_timing_missing_pct": round(float(missing_job_timing), 2),
    "job_negative_durations": int(negative),
    "negative_by_conclusion": jobs_df[jobs_df.duration < 0].conclusion.value_counts().to_dict(),
    "negative_median_s": float(jobs_df[jobs_df.duration < 0].duration.median()) if negative else 0.0,
    "step_timing_missing_pct": round(float(step_missing), 2),
    "skipped_jobs_with_zero_duration": int(((jobs_df.conclusion == "skipped") & (jobs_df.duration.fillna(0) <= 1)).sum()),
}
R["retried_runs"] = int((runs_df.run_attempt > 1).sum())
print(per_repo)
print(R["job_conclusion_pct"], R["missing"], R["event_pct"])

## b. Distribution analysis

Series = successful executions of one job (repo x job display name), chronological.

In [ ]:
succ = jobs_df[(jobs_df.conclusion == "success") & jobs_df.duration.notna() & (jobs_df.duration > 0)]
series = {k: g.sort_values("created_at").duration.to_numpy() for k, g in succ.groupby(["repo", "name"]) if len(g) >= 20}
R["n_series_20"] = len(series)

skews = np.array([stats.skew(v) for v in series.values()])
log_skews = np.array([stats.skew(np.log(v)) for v in series.values()])
mean_over_median = np.array([v.mean() / np.median(v) for v in series.values()])
R["skew"] = {
    "median_skew": round(float(np.median(skews)), 2),
    "pct_skew_gt_1": round(float((skews > 1).mean() * 100), 1),
    "median_log_skew": round(float(np.median(log_skews)), 2),
    "pct_mean_gt_median_10pct": round(float((mean_over_median > 1.10).mean() * 100), 1),
    "pct_shapiro_reject_normal": round(
        float(np.mean([stats.shapiro(v[:500]).pvalue < 0.05 for v in series.values()]) * 100), 1
    ),
}

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
pooled = np.concatenate([v / np.median(v) for v in series.values()])
axes[0].hist(np.clip(pooled, 0, 4), bins=80, color="#4C72B0")
axes[0].axvline(1, color="k", lw=0.8)
axes[0].set(title="Job duration / its own median (pooled)", xlabel="ratio (clipped at 4)", ylabel="job executions")
axes[1].hist(np.clip(skews, -2, 10), bins=40, color="#4C72B0", alpha=0.8, label="raw seconds")
axes[1].hist(np.clip(log_skews, -2, 10), bins=40, color="#DD8452", alpha=0.6, label="log seconds")
axes[1].axvline(0, color="k", lw=0.8)
axes[1].set(title="Per-series sample skewness", xlabel="skewness (clipped)", ylabel="series")
axes[1].legend(frameon=False)
savefig("b1_skew")
print(R["skew"])

### Threshold estimators: mean+2·SD vs median+2·(1.4826·MAD) vs empirical 97.7th percentile

All three target the same one-sided tail (2.3% under normality). Fit on the first 2/3 of each
series, measure the exceedance rate on the last 1/3 (calibration) and the bootstrap spread of
the threshold relative to the median (stability).

In [ ]:
def thr_mean(x):
    return x.mean() + 2 * x.std(ddof=1)


def thr_mad(x):
    return np.median(x) + 2 * 1.4826 * stats.median_abs_deviation(x)


def thr_pct(x):
    return np.percentile(x, 97.7)


ESTIMATORS = {"mean+2sd": thr_mean, "median+2MAD": thr_mad, "p97.7": thr_pct}
rng = np.random.default_rng(SEED)
rows = []
for key, v in series.items():
    if len(v) < 30:
        continue
    cut = len(v) * 2 // 3
    train, test = v[:cut], v[cut:]
    for name, f in ESTIMATORS.items():
        t = f(train)
        boots = [f(rng.choice(train, size=len(train), replace=True)) for _ in range(200)]
        rows.append(
            {
                "series": key,
                "estimator": name,
                "exceed_test": float((test > t).mean()),
                "rel_boot_sd": float(np.std(boots) / np.median(train)),
                "thr_over_median": float(t / np.median(train)),
            }
        )
thr = pd.DataFrame(rows)
thr_summary = thr.groupby("estimator").agg(
    series=("series", "nunique"),
    median_exceed_pct=("exceed_test", lambda s: round(s.median() * 100, 1)),
    mean_exceed_pct=("exceed_test", lambda s: round(s.mean() * 100, 1)),
    pct_series_exceed_over_10pct=("exceed_test", lambda s: round((s > 0.10).mean() * 100, 1)),
    median_rel_boot_sd=("rel_boot_sd", lambda s: round(s.median(), 3)),
    median_thr_over_median=("thr_over_median", lambda s: round(s.median(), 2)),
)
R["thr_table"] = thr_summary.to_markdown()
R["thr"] = thr_summary.to_dict(orient="index")

fig, ax = plt.subplots(figsize=(6, 3.2))
for name in ESTIMATORS:
    s = np.sort(thr[thr.estimator == name].exceed_test.to_numpy())
    ax.plot(s, np.linspace(0, 1, len(s)), label=name)
ax.axvline(0.023, color="k", ls="--", lw=0.8, label="nominal 2.3%")
ax.set(xlim=(0, 0.4), xlabel="exceedance rate on held-out runs", ylabel="cumulative share of series",
       title="Out-of-sample calibration of slow-run thresholds")
ax.legend(frameon=False)
savefig("b2_threshold_calibration")
print(thr_summary)

### Coefficient of variation (SD/mean) vs a robust analogue (1.4826·MAD/median)

Several analyzers branch on CV: savings are `unknown` when CV < 0.10, the "high variance"
recommendation fires at CV >= 0.25, and Dependency Install calls a cached step
"possibly ineffective" at CV >= 0.30. One slow outlier can push SD/mean over these lines.

In [ ]:
def robust_cv(x):
    m = np.median(x)
    return 1.4826 * stats.median_abs_deviation(x) / m if m > 0 else np.nan


cv_rows = [{"series": k, "cv": np.std(v, ddof=1) / np.mean(v), "rcv": robust_cv(v)} for k, v in series.items()]
cvdf = pd.DataFrame(cv_rows)
R["cv"] = {}
for t in (0.10, 0.25, 0.30):
    a, b = cvdf.cv >= t, cvdf.rcv >= t
    R["cv"][str(t)] = {
        "cv_ge": round(float(a.mean() * 100), 1),
        "robust_ge": round(float(b.mean() * 100), 1),
        "disagree": round(float((a != b).mean() * 100), 1),
        "cv_only": int((a & ~b).sum()),
        "robust_only": int((~a & b).sum()),
    }
# Outlier sensitivity: add one run at 3x the median to every series.
bumped = [
    (np.std(np.append(v, 3 * np.median(v)), ddof=1) / np.mean(np.append(v, 3 * np.median(v))) - np.std(v, ddof=1) / np.mean(v),
     robust_cv(np.append(v, 3 * np.median(v))) - robust_cv(v))
    for v in series.values()
]
bumped = np.array(bumped)
R["cv_one_outlier_median_shift"] = {"cv": round(float(np.median(bumped[:, 0])), 3), "robust": round(float(np.median(bumped[:, 1])), 3)}

fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.scatter(cvdf.rcv.clip(0, 1.2), cvdf.cv.clip(0, 1.2), s=6, alpha=0.5)
ax.plot([0, 1.2], [0, 1.2], color="k", lw=0.7)
for t in (0.10, 0.30):
    ax.axhline(t, color="grey", ls=":", lw=0.7)
    ax.axvline(t, color="grey", ls=":", lw=0.7)
ax.set(xlabel="robust CV (1.4826·MAD/median)", ylabel="CV (SD/mean)", title="CV vs robust CV per job series")
savefig("b3_cv_vs_robust")
print(R["cv"], R["cv_one_outlier_median_shift"])

## c. Sensitivity analysis

Each analyzer is run once per repo with its gates opened as far as its metrics allow; the
threshold grid is then applied to the metrics each finding reports. Defaults are marked.

In [ ]:
def run_all(analyzer):
    out = []
    for repo in REPOS:
        for f in analyzer.analyze(RUNS[repo]):
            out.append({"repo": repo, **f.metrics, "_finding": f})
    return pd.DataFrame(out)


loose_job = run_all(SlowJobAnalyzer(SlowJobConfig(min_duration_seconds=0, absolute_severe_seconds=1e12, min_share_of_workflow=0)))
loose_step = run_all(SlowStepAnalyzer(SlowStepConfig(min_duration_seconds=0, absolute_severe_seconds=1e12, min_share_of_job=0)))
from adpo.analyzers.historical_regression import RegressionConfig  # noqa: E402

loose_reg = run_all(HistoricalRegressionAnalyzer(RegressionConfig(min_pct_increase=-1e9, min_absolute_increase_seconds=-1e9)))
loose_dep = run_all(DependencyInstallAnalyzer(DependencyInstallConfig(min_duration_seconds=0, ineffective_cache_cv_threshold=0)))
loose_fail = run_all(FailureRetryAnalyzer(FailureRetryConfig(flaky_min_rate=0)))
loose_par = run_all(ParallelizationAnalyzer(ParallelizationConfig(min_job_duration_seconds=0, min_sequential_consistency=0)))

d_job, d_step = SlowJobConfig(), SlowStepConfig()


def count_job(share, n_min=1):
    if loose_job.empty:
        return 0
    m, s = loose_job.median_duration_seconds, loose_job.share_of_workflow_median
    hit = ((m >= d_job.min_duration_seconds) & (s >= share)) | (m >= d_job.absolute_severe_seconds)
    return int((hit & (loose_job.sample_size >= n_min)).sum())


def count_step(share, n_min=1):
    if loose_step.empty:
        return 0
    m, s = loose_step.median_duration_seconds, loose_step.share_of_job_median.fillna(-1)
    hit = ((m >= d_step.min_duration_seconds) & (s >= share)) | (m >= d_step.absolute_severe_seconds)
    return int((hit & (loose_step.sample_size >= n_min)).sum())


def count_reg(pct):
    if loose_reg.empty:
        return 0
    return int(((loose_reg.pct_increase >= pct) & (loose_reg.absolute_increase_seconds >= 20)).sum())


def count_dep(cv_t):
    if loose_dep.empty:
        return 0
    cached = loose_dep.cache_action_detected.astype(bool)
    keep = (loose_dep.median_duration_seconds >= 20) & (~cached | (loose_dep.coefficient_of_variation.fillna(0) >= cv_t))
    return int(keep.sum())


flaky = loose_fail[loose_fail["classification"].isin(["flaky", "chronic_failure"])] if len(loose_fail) else loose_fail


def count_flaky(rate):
    if flaky.empty:
        return 0
    return int((flaky.failure_rate >= rate).sum())


def count_par(cons):
    if loose_par.empty:
        return 0
    ok = (loose_par.job_a_median_seconds >= 20) & (loose_par.job_b_median_seconds >= 20)
    return int((ok & (loose_par.sequential_consistency >= cons)).sum())


grids = {
    "slow_job: min_share_of_workflow": (np.round(np.arange(0.05, 0.61, 0.05), 2), count_job, 0.25),
    "slow_step: min_share_of_job": (np.round(np.arange(0.1, 0.91, 0.1), 2), count_step, 0.4),
    "historical_regression: min_pct_increase": (np.round(np.arange(0.05, 0.81, 0.05), 2), count_reg, 0.25),
    "dependency_install: cache CV threshold": (np.round(np.arange(0.1, 0.81, 0.1), 2), count_dep, 0.30),
    "failure_retry: flaky_min_rate": (np.round([0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3], 2), count_flaky, 0.05),
    "parallelization: min_sequential_consistency": (np.round(np.arange(0.5, 1.01, 0.05), 2), count_par, 0.75),
}
fig, axes = plt.subplots(2, 3, figsize=(10, 5.4))
R["sensitivity"] = {}
for ax, (title, (grid, fn, default)) in zip(axes.flat, grids.items()):
    ys = [fn(g) for g in grid]
    R["sensitivity"][title] = dict(zip([float(g) for g in grid], ys))
    ax.plot(grid, ys, marker="o", ms=3)
    ax.axvline(default, color="crimson", ls="--", lw=0.8)
    ax.set(title=title, xlabel="threshold", ylabel="findings (all repos)")
savefig("c1_sensitivity")

# Small-sample findings at defaults.
R["small_n"] = {
    "slow_job_defaults": count_job(0.25),
    "slow_job_n_lt_5": count_job(0.25) - count_job(0.25, n_min=5),
    "slow_step_defaults": count_step(0.4),
    "slow_step_n_lt_5": count_step(0.4) - count_step(0.4, n_min=5),
}
# Parallelization: how much of the signal depends on the YAML `needs` edges.
par_no_yaml = []
for repo in REPOS:
    stripped = to_raw_runs(tables, repo)
    for r in stripped:
        for j in r["jobs"]:
            j["needs"] = []
    par_no_yaml += ParallelizationAnalyzer().analyze(parse_workflow_runs(stripped))
R["par_with_yaml"] = count_par(0.75)
R["par_without_yaml"] = len(par_no_yaml)
print(json.dumps(R["sensitivity"], indent=1)[:1500], R["small_n"], R["par_with_yaml"], R["par_without_yaml"])

## d. Regression detection: ADPO window rule vs Mann-Whitney U vs change-point

Same series as (b) with n >= 12 successful runs. Methods:
* **ADPO**: recent 40% median vs baseline median, flagged if +25% and +20 s.
* **MWU**: one-sided Mann-Whitney U (recent > baseline) on the same windows, Benjamini-Hochberg q < 0.05.
* **MWU+effect**: MWU significant *and* the ADPO effect-size gates.
* **Change-point**: PELT (L1 cost, log durations, penalty 2·log(n)·σ̂ (BIC-style for an L1/Laplace cost) with σ̂ from the MAD of
  successive differences); flagged if the last segment's median is >= 25% and >= 20 s above the
  previous segment's, and the last segment has >= 3 runs.

In [ ]:
reg_an = HistoricalRegressionAnalyzer()
cfg = reg_an.config


def windows(v):
    n = len(v)
    recent_n = min(max(cfg.min_recent_samples, round(n * cfg.recent_fraction)), n - cfg.min_baseline_samples)
    return v[: n - recent_n], v[n - recent_n :]


def changepoint(v):
    x = np.log(v)
    sigma = 1.4826 * stats.median_abs_deviation(np.diff(x)) / math.sqrt(2) or 1e-6
    bkps = rpt.Pelt(model="l1", min_size=3, jump=1).fit(x).predict(pen=2 * math.log(len(x)) * sigma)
    if len(bkps) < 2:
        return False, bkps
    prev = v[(bkps[-3] if len(bkps) >= 3 else 0) : bkps[-2]]
    last = v[bkps[-2] :]
    up = np.median(last) - np.median(prev)
    return bool(len(last) >= 3 and up >= 20 and up / np.median(prev) >= 0.25), bkps


reg_series = {k: v for k, v in
              ((k, g.sort_values("created_at").duration.to_numpy()) for k, g in succ.groupby(["repo", "name"]))
              if len(v) >= 12}
rows = []
for key, v in reg_series.items():
    base, rec = windows(v)
    adpo = reg_an._analyze_series(str(key), [(float(d), str(i)) for i, d in enumerate(v)]) is not None
    p = stats.mannwhitneyu(rec, base, alternative="greater").pvalue
    eff = (np.median(rec) - np.median(base) >= 20) and (np.median(rec) / np.median(base) - 1 >= 0.25)
    cp, bkps = changepoint(v)
    rows.append({"series": key, "n": len(v), "adpo": adpo, "p": p, "effect": eff, "cp": cp, "bkps": bkps,
                 "pct": np.median(rec) / np.median(base) - 1})
reg = pd.DataFrame(rows)
reg["q"] = stats.false_discovery_control(reg.p.to_numpy(), method="bh")
reg["mwu"] = reg.q < 0.05
reg["mwu_eff"] = reg.mwu & reg.effect


def kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    po = (a == b).mean()
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return round(float((po - pe) / (1 - pe)), 3) if pe < 1 else float("nan")


methods = ["adpo", "mwu", "mwu_eff", "cp"]
R["reg_n_series"] = len(reg)
R["reg_flags"] = {m: int(reg[m].sum()) for m in methods}
R["reg_kappa"] = {f"{a} vs {b}": kappa(reg[a], reg[b]) for i, a in enumerate(methods) for b in methods[i + 1 :]}
R["reg_confusion_adpo_mwu"] = pd.crosstab(reg.adpo, reg.mwu, rownames=["ADPO"], colnames=["MWU q<0.05"]).to_markdown()
R["reg_adpo_not_mwu"] = int((reg.adpo & ~reg.mwu).sum())
R["reg_mwu_eff_not_adpo"] = int((reg.mwu_eff & ~reg.adpo).sum())
R["reg_adpo_not_cp"] = int((reg.adpo & ~reg.cp).sum())
R["reg_cp_not_adpo"] = int((reg.cp & ~reg.adpo).sum())


def pick(mask, k=1):
    sub = reg[mask].sort_values("n", ascending=False)
    return sub.head(k).to_dict("records")


examples = {
    "ADPO flags, MWU does not": pick(reg.adpo & ~reg.mwu),
    "MWU+effect flags, ADPO does not": pick(reg.mwu_eff & ~reg.adpo),
    "Change-point flags, ADPO does not": pick(reg.cp & ~reg.adpo),
    "All methods agree (flag)": pick(reg.adpo & reg.mwu & reg.cp),
}
fig, axes = plt.subplots(2, 2, figsize=(10, 5.6))
R["reg_examples"] = []
for ax, (label, ex) in zip(axes.flat, examples.items()):
    ax.set_title(label, fontsize=9)
    if not ex:
        ax.text(0.5, 0.5, "no such series", ha="center", transform=ax.transAxes)
        continue
    e = ex[0]
    v = reg_series[e["series"]]
    base, rec = windows(v)
    ax.plot(v, ".", ms=3, color="#4C72B0")
    ax.hlines(np.median(base), 0, len(base) - 1, color="k", lw=1)
    ax.hlines(np.median(rec), len(base), len(v) - 1, color="crimson", lw=1)
    for b in e["bkps"][:-1]:
        ax.axvline(b, color="orange", ls="--", lw=0.8)
    ax.set(xlabel="run (chronological)", ylabel="seconds")
    ax.text(0.01, 0.95, f"{e['series'][0]}\n{e['series'][1][:45]}\nq={e['q']:.3f}, Δmedian={e['pct']*100:+.0f}%",
            transform=ax.transAxes, va="top", fontsize=7)
    R["reg_examples"].append({"case": label, "repo": e["series"][0], "job": e["series"][1], "n": int(e["n"]),
                              "q": round(float(e["q"]), 4), "pct": round(float(e["pct"]) * 100, 1),
                              "adpo": bool(e["adpo"]), "mwu": bool(e["mwu"]), "cp": bool(e["cp"])})
savefig("d1_regression_examples")
print(R["reg_flags"], R["reg_kappa"])
print(R["reg_confusion_adpo_mwu"])

## e. Flakiness: Beta-Binomial rates, a re-run ground truth, and co-failures

**Rates.** Per job, n = runs in which it reached success/failure on its final attempt and
k = final-attempt failures (ADPO's flaky signal) or k = flake events (below). Prior: Jeffreys
Beta(0.5, 0.5); posterior Beta(0.5+k, 0.5+n−k); 95% equal-tailed credible interval. Jobs with
fewer than `N_MIN` runs get no rate and no rank: with Jeffreys, n = 0 gives a posterior mean of
0.5, and small n gives intervals too wide to rank on. `N_MIN` is chosen below from split-half
reliability.

**Ground truth.** A *flake event* is a run (one head_sha) in which a job failed on an earlier
attempt and passed on a later attempt (run_attempt > 1). A job is *re-run-confirmed flaky* if it
has at least one flake event. This truth is incomplete: jobs nobody re-ran can't be confirmed, so
precision against it is a lower bound. It is reported separately for pull-request runs and for
push-to-default-branch runs.

**Co-failure.** For each failing execution: the share of the *other* jobs in the same attempt
that also failed. Failures shared with most of the run point at the commit or the environment
rather than the job.

In [ ]:
DEFAULT_BRANCH = {r: meta["repos"][r].get("default_branch") for r in REPOS}
R["default_branch_known"] = sum(v is not None for v in DEFAULT_BRANCH.values())
att = jobs_df[jobs_df.conclusion.isin(["success", "failure"])].copy()
att["run_attempt"] = att.run_attempt.astype(int)
att = att.merge(runs_df[["repo", "run_id", "head_sha", "run_number"]], on=["repo", "run_id"])
att["subset"] = np.where(
    att.event == "pull_request",
    "pull_request",
    np.where((att.event == "push") & (att.branch == att.repo.map(DEFAULT_BRANCH)), "push_default", "other"),
)
assert att.groupby(["repo", "base_run"]).head_sha.nunique().max() == 1  # attempts of a run share one SHA

# Co-failure share of the other jobs in the same attempt.
per_attempt = att.assign(f=att.conclusion == "failure").groupby(["repo", "run_id"]).f.agg(["sum", "size"])
att = att.join(per_attempt, on=["repo", "run_id"])
att["others_failed_share"] = np.where(
    att["size"] > 1, (att["sum"] - (att.conclusion == "failure")) / (att["size"] - 1).clip(lower=1), 0.0
)


def per_run_table(a):
    """One row per (job, run): outcome on the run's FINAL attempt, exactly as FailureRetryAnalyzer
    sees it (a job cancelled/skipped or absent on the final attempt is not counted), plus whether the
    job had a flake event (failed on an earlier attempt, passed on a later one) in that run."""
    key = ["repo", "subset", "name", "base_run"]
    final_attempt = runs_df.groupby(["repo", "base_run"]).run_attempt.max().rename("final_attempt")
    a = a.join(final_attempt, on=["repo", "base_run"])
    last = a[a.run_attempt == a.final_attempt].set_index(key)
    first_fail = a[a.conclusion == "failure"].groupby(key).run_attempt.min()
    last_pass = a[a.conclusion == "success"].groupby(key).run_attempt.max()
    t = pd.DataFrame({
        "final_failed": last.conclusion == "failure",
        "cofail": last.others_failed_share >= 0.5,
        "attempts": a.groupby(key).run_attempt.nunique().reindex(last.index),
    })
    t["flake"] = (last_pass.reindex(t.index) > first_fail.reindex(t.index)).fillna(False).astype(bool)
    return t.reset_index()


prt = per_run_table(att)
R["flake_events_by_subset"] = prt.groupby("subset").flake.sum().astype(int).to_dict()
R["reruns_by_subset"] = (prt.attempts > 1).groupby(prt.subset).sum().astype(int).to_dict()
R["runs_by_subset"] = prt.groupby("subset").base_run.nunique().astype(int).to_dict()


def jeffreys(k, n):
    a, b = 0.5 + k, 0.5 + n - k
    return a / (a + b), stats.beta.ppf(0.025, a, b), stats.beta.ppf(0.975, a, b), stats.beta.ppf(0.05, a, b)


# ---- N_MIN from split-half reliability ----------------------------------------------------
# For each job with >= 2N runs, draw two disjoint random sets of N runs (seeded) and estimate the
# failure rate on each; the Spearman correlation of the two estimates across jobs is the
# reliability of a rate computed from N runs. N_MIN = smallest N reaching 0.7 (a conventional
# "acceptable reliability" level). Averaged over 20 seeded draws.
rel_rng = np.random.default_rng(SEED)
fails_by_job = [g.final_failed.to_numpy() for _, g in prt.groupby(["repo", "name"])]
rel_rows = []
for N in (3, 5, 8, 10, 15, 20, 30, 40, 50):
    pool = [f for f in fails_by_job if len(f) >= 2 * N]
    rhos = []
    for _ in range(20):
        xs, ys = [], []
        for f in pool:
            idx = rel_rng.permutation(len(f))
            xs.append(f[idx[:N]].mean())
            ys.append(f[idx[N : 2 * N]].mean())
        if len(xs) > 5 and np.std(xs) > 0 and np.std(ys) > 0:
            rhos.append(stats.spearmanr(xs, ys).statistic)
    lo, hi = jeffreys(1, N)[1:3]
    rel_rows.append({"N": N, "jobs_with_2N_runs": len(pool),
                     "split_half_spearman": round(float(np.mean(rhos)), 3) if rhos else np.nan,
                     "jeffreys_ci95_width_at_1_failure": round(float(hi - lo), 3)})
rel = pd.DataFrame(rel_rows)
ok = rel[rel.split_half_spearman >= 0.7]
# Fallback if reliability 0.7 is never reached: smallest N whose 95% Jeffreys interval is <= 0.25 wide
# at the median observed failure rate (expected k = p*N, so the width is a smooth function of N).
p_med = float(np.median([f.mean() for f in fails_by_job]))
widths = {N: float(stats.beta.ppf(0.975, 0.5 + p_med * N, 0.5 + (1 - p_med) * N)
                   - stats.beta.ppf(0.025, 0.5 + p_med * N, 0.5 + (1 - p_med) * N)) for N in range(1, 501)}
if len(ok):
    N_MIN, N_MIN_RULE = int(ok.N.min()), "split-half reliability >= 0.7"
else:
    N_MIN, N_MIN_RULE = min(N for N, w in widths.items() if w <= 0.25), "fallback: Jeffreys 95% width <= 0.25 at median rate"
R["n_min_rule"] = N_MIN_RULE
R["median_failure_rate"] = round(p_med, 4)
R["max_split_half"] = float(rel.split_half_spearman.max())
R["n_min_table"] = rel.to_markdown(index=False)
R["N_MIN"] = N_MIN
print(rel, "N_MIN =", N_MIN, N_MIN_RULE, "median rate", p_med)

### Ranking before/after the posterior (jobs with n >= N_MIN only)

In [ ]:
jt = prt.groupby(["repo", "name"]).agg(n=("final_failed", "size"), fails=("final_failed", "sum"),
                                       flakes=("flake", "sum")).reset_index()
R["jobs_below_n_min"] = int((jt.n < N_MIN).sum())
R["jobs_total"] = len(jt)
elig = jt[jt.n >= N_MIN].copy()
elig["raw"] = elig.fails / elig.n
elig["post"], elig["lo"], elig["hi"], elig["lb90"] = jeffreys(elig.fails, elig.n)
fmt = lambda d: d.assign(job=d.name.str[:40], raw_pct=(d.raw * 100).round(1), post_pct=(d.post * 100).round(1),
                         ci95=(d.lo * 100).round(1).astype(str) + "–" + (d.hi * 100).round(1).astype(str))[
    ["repo", "job", "n", "fails", "raw_pct", "post_pct", "ci95"]]
top_raw = elig.sort_values(["raw", "n"], ascending=[False, False]).head(10)
top_post = elig.sort_values("lb90", ascending=False).head(10)
R["fail_top_raw"] = fmt(top_raw).to_markdown(index=False)
R["fail_top_post"] = fmt(top_post).to_markdown(index=False)
R["fail_rank_overlap_top10"] = len(set(map(tuple, top_raw[["repo", "name"]].values))
                                   & set(map(tuple, top_post[["repo", "name"]].values)))

fl_el = elig.copy()
fl_el["raw"] = fl_el.flakes / fl_el.n
fl_el["post"], fl_el["lo"], fl_el["hi"], fl_el["lb90"] = jeffreys(fl_el.flakes, fl_el.n)
R["flake_jobs_with_event"] = int((jt.flakes > 0).sum())
R["flake_top_post"] = (fmt(fl_el[fl_el.flakes > 0].sort_values("lb90", ascending=False).head(10).rename(
    columns={"flakes": "fails"})).rename(columns={"fails": "flake_events"}).to_markdown(index=False))

fig, ax = plt.subplots(figsize=(6.8, 3.8))
show = elig[elig.fails > 0].sort_values("raw", ascending=False).head(25).iloc[::-1]
y = np.arange(len(show))
ax.errorbar(show.post * 100, y, xerr=[(show.post - show.lo) * 100, (show.hi - show.post) * 100],
            fmt="o", ms=3, color="#4C72B0", label="Jeffreys posterior mean, 95% CrI")
ax.scatter(show.raw * 100, y, marker="x", color="crimson", s=14, label="raw rate")
ax.set_yticks(y, [f"{r.split('/')[1]}: {j[:28]} (n={n})" for r, j, n in zip(show.repo, show.name, show.n)], fontsize=6)
ax.set(xlabel="final-attempt failure rate (%)", title=f"Top 25 jobs by raw failure rate (n ≥ {N_MIN})")
ax.legend(frameon=False, fontsize=7)
savefig("e1_failure_posterior")

### ADPO's flaky flags vs the re-run ground truth, by trigger

`original` is the rule ADPO shipped before this evaluation (n >= 3, final-attempt failure rate
>= 5%, below the 85% chronic line). Variants re-apply it to the same final-attempt outcomes:
`jeffreys_gate` needs n >= N_MIN and a one-sided 95% Jeffreys lower bound (5th percentile of
Beta(k+0.5, n−k+0.5)) >= 5%; `drop_cofail` removes runs whose failing final attempt had >= 50% of
the other jobs also failing; `both` combines them. The `original` simulation is asserted equal to
the real `FailureRetryAnalyzer` on every subset, so the variants differ from it only by their rule.

**Adoption rule (stated before the final table):** a variant replaces `original` only if, on PR
runs, precision rises by >= 0.10 with non-overlapping Wilson intervals AND recall falls by <= 0.20
(absolute), AND it loses no confirmed-flaky job on push-to-default runs.

In [ ]:
cfgf = FailureRetryConfig()
LEGACY = cfgf  # the shipped rule is the original rule (see the adoption decision below)


def rule(g, variant):
    if variant in ("drop_cofail", "both"):
        g = g[~(g.final_failed & g.cofail)]
    n, k = len(g), int(g.final_failed.sum())
    if n == 0 or k == 0:
        return False
    rate = k / n
    if rate >= cfgf.chronic_failure_rate:
        return False
    if variant in ("jeffreys_gate", "both"):
        return n >= N_MIN and stats.beta.ppf(0.05, 0.5 + k, 0.5 + n - k) >= cfgf.flaky_min_rate
    return n >= LEGACY.min_sample_size and rate >= LEGACY.flaky_min_rate


def subset_runs(repo, subset):
    keep = []
    for r in RUNS[repo]:
        s = "pull_request" if r.event == "pull_request" else (
            "push_default" if r.event == "push" and r.branch == DEFAULT_BRANCH[repo] else "other")
        if s == subset:
            keep.append(r)
    return keep


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))


VARIANTS = ["original", "jeffreys_gate", "drop_cofail", "both"]
pr_rows = []
for subset in ("push_default", "pull_request"):
    sp = prt[prt.subset == subset]
    groups = {key: g for key, g in sp.groupby(["repo", "name"])}
    truth = {key for key, g in groups.items() if g.flake.any()}
    rerun_seen = {key for key, g in groups.items() if (g.attempts > 1).any()}
    def analyzer_flags(cfg):
        out = set()
        for repo in REPOS:
            for f in FailureRetryAnalyzer(cfg).analyze(subset_runs(repo, subset)):
                if f.metrics.get("classification") == "flaky":
                    out.add((repo, f.metrics["job_name"]))
        return out

    flagged = {v: {key for key, g in groups.items() if rule(g, v)} for v in VARIANTS}
    assert flagged["original"] == analyzer_flags(LEGACY), subset

    for v in VARIANTS:
        fset = flagged[v]
        tp = len(fset & truth)
        p_lo, p_hi = wilson(tp, len(fset))
        r_lo, r_hi = wilson(tp, len(truth))
        obs = fset & rerun_seen
        pr_rows.append({
            "subset": subset, "variant": v, "flagged": len(fset), "confirmed_flaky": len(truth), "tp": tp,
            "precision_lb": round(tp / len(fset), 3) if fset else np.nan,
            "precision_95ci": f"{p_lo:.2f}–{p_hi:.2f}" if fset else "",
            "precision_among_rerun_jobs": round(len(obs & truth) / len(obs), 3) if obs else np.nan,
            "recall": round(tp / len(truth), 3) if truth else np.nan,
            "recall_95ci": f"{r_lo:.2f}–{r_hi:.2f}" if truth else "",
        })
prdf = pd.DataFrame(pr_rows)


def adopt(v):
    o = prdf.set_index(["subset", "variant"])
    po, pv = o.loc[("pull_request", "original")], o.loc[("pull_request", v)]
    lo_v = float(pv.precision_95ci.split("–")[0]) if pv.precision_95ci else float("nan")
    hi_o = float(po.precision_95ci.split("–")[1])
    gain = pv.precision_lb - po.precision_lb
    drop = po.recall - pv.recall
    push_loss = o.loc[("push_default", "original")].tp - o.loc[("push_default", v)].tp
    ok = bool(gain >= 0.10 and lo_v > hi_o and drop <= 0.20 and push_loss <= 0)
    return {"adopt": ok, "precision_gain": round(float(gain), 3), "recall_drop": round(float(drop), 3),
            "ci_separated": bool(lo_v > hi_o), "push_confirmed_lost": int(push_loss)}


R["adoption"] = {v: adopt(v) for v in VARIANTS if v != "original"}
R["flaky_pr_table"] = prdf.to_markdown(index=False)
R["flaky_pr"] = prdf.set_index(["subset", "variant"]).to_dict(orient="index")
print(prdf)

### Co-failure across all repos

In [ ]:
cur_flag = set()
for subset in ("push_default", "pull_request"):
    sp = prt[prt.subset == subset]
    cur_flag |= {(subset, *key) for key, g in sp.groupby(["repo", "name"]) if rule(g, "original")}
ff = prt[prt.final_failed].copy()
ff["flagged"] = [(s, r, n) in cur_flag for s, r, n in zip(ff.subset, ff.repo, ff.name)]
fx = ff[ff.flagged]
R["cofail"] = {
    "failing_executions_of_flagged_jobs": len(fx),
    "pct_cofail_ge_half": round(float(fx.cofail.mean() * 100), 1) if len(fx) else None,
    "pct_cofail_by_subset": (fx.groupby("subset").cofail.mean() * 100).round(1).to_dict(),
    "flagged_jobs": len(cur_flag),
    "flagged_jobs_majority_cofail": int((fx.groupby(["subset", "repo", "name"]).cofail.mean() > 0.5).sum()),
    "repos_with_flagged_jobs": int(fx.repo.nunique()),
    "repos_majority_cofail": int((fx.groupby("repo").cofail.mean() > 0.5).sum()),
}
cof_repo = (fx.groupby("repo").cofail.mean() * 100).round(0)
R["cofail_by_repo"] = cof_repo.to_dict()
ev = att.groupby("subset").apply(lambda g: (g.conclusion == "failure").mean() * 100, include_groups=False).round(1)
R["fail_rate_by_subset_pct"] = ev.to_dict()

fig, ax = plt.subplots(figsize=(6, 3.4))
cof_repo.sort_values().plot.barh(ax=ax, color="#4C72B0")
ax.axvline(50, color="k", lw=0.7, ls="--")
ax.set(xlabel="% of flagged jobs' failing runs where ≥ half the other jobs also failed", ylabel="",
       title="Co-failure share, ADPO-flagged 'flaky' jobs")
ax.tick_params(axis="y", labelsize=7)
savefig("e2_cofailure_by_repo")
print(R["cofail"], R["fail_rate_by_subset_pct"])

print(R["adoption"])

## f. Precision check (hand labels)

All findings at default settings, sampled with a fixed seed, stratified so each analyzer that
produced findings is represented (40 total, split as evenly as availability allows). Labels are
filled in by hand in `labels.csv` (useful / not useful / wrong); the next cell computes precision.

In [ ]:
engine = AnalysisEngine()
all_f = []
for repo in REPOS:
    for f in engine.run(RUNS[repo]):
        all_f.append({"repo": repo, "analyzer": f.analyzer_type, "title": f.title, "f": f})
    assert not engine.errors, engine.errors
all_f = sorted(all_f, key=lambda d: (d["repo"], d["analyzer"], d["title"]))
for i, d in enumerate(all_f):
    d["finding_id"] = f"F{i:04d}"
fdf = pd.DataFrame(all_f)
R["findings_default"] = fdf.groupby("analyzer").size().to_dict()
R["findings_default_total"] = len(fdf)
R["findings_per_repo"] = fdf.groupby("repo").size().reindex(REPOS, fill_value=0).to_dict()

rng = np.random.default_rng(SEED)
avail = fdf.groupby("analyzer").size().to_dict()
alloc = {a: 0 for a in avail}
remaining = 40
while remaining > 0 and any(alloc[a] < avail[a] for a in avail):
    for a in sorted(avail):
        if remaining and alloc[a] < avail[a]:
            alloc[a] += 1
            remaining -= 1
sample = pd.concat([fdf[fdf.analyzer == a].iloc[rng.choice(avail[a], alloc[a], replace=False)] for a in sorted(alloc)])
R["label_alloc"] = alloc

labels_path = HERE / "labels.csv"
existing = pd.read_csv(labels_path, dtype=str).fillna("") if labels_path.exists() else None
has_labels = existing is not None and (existing.label.str.strip() != "").any()
if has_labels and set(existing.finding_id) != set(sample.finding_id):
    raise RuntimeError("labels.csv holds hand labels for a different sample (different data source?); refusing to overwrite it")
if existing is None or set(existing.finding_id) != set(sample.finding_id):
    pd.DataFrame({
        "finding_id": sample.finding_id,
        "repo": sample.repo,
        "analyzer": sample.analyzer,
        "evidence_summary": [d.title + " | " + " | ".join(d.evidence[:3]) for d in sample.f],
        "label": "",
        "notes": "",
    }).to_csv(labels_path, index=False)
(HERE / "data" / "sampled_findings.json").write_text(json.dumps(
    [{"finding_id": i, "repo": r, **f.to_dict()} for i, r, f in zip(sample.finding_id, sample.repo, sample.f)],
    indent=1, default=str) + "\n")
print(alloc, len(sample))

In [ ]:
lab = pd.read_csv(labels_path, dtype=str).fillna("")
lab["label"] = lab.label.str.strip().str.lower()
done = lab[lab.label != ""]
if done.empty:
    R["precision_table"] = "_Not computed yet: `labels.csv` has no labels. Fill in the `label` column and re-run._"
else:
    rows = []
    for a, g in list(done.groupby("analyzer")) + [("ALL", done)]:
        k, n = int((g.label == "useful").sum()), len(g)
        lo, hi = wilson(k, n)
        rows.append({"analyzer": a, "labelled": n, "useful": k, "precision": round(k / n, 2),
                     "wilson_95_lo": round(lo, 2), "wilson_95_hi": round(hi, 2)})
    R["precision_table"] = pd.DataFrame(rows).to_markdown(index=False)
print(R["precision_table"])

## Write REPORT.md

Everything below is filled from `R`; claims whose direction depends on the data are computed.

In [ ]:
def fp(x, nd=1):
    return "n/a" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{x:.{nd}f}"


T = R["totals"]
thr = R["thr"]
nominal = 2.3
closest = min(thr, key=lambda k: abs(thr[k]["median_exceed_pct"] - nominal))
closest_mean = min(thr, key=lambda k: abs(thr[k]["mean_exceed_pct"] - nominal))
worst_tail = max(thr, key=lambda k: thr[k]["pct_series_exceed_over_10pct"])
pr = R["flaky_pr"]
P = lambda sub, v, k: pr[(sub, v)][k]  # noqa: E731
sens = R["sensitivity"]
reg = R["reg_flags"]
co = R["cofail"]
ex_lines = "\n".join(
    f"- *{e['case']}*: `{e['repo']}` / `{e['job']}` (n={e['n']}, recent vs baseline median {e['pct']:+.0f}%, "
    f"BH q={e['q']:.3f}; ADPO={e['adpo']}, MWU={e['mwu']}, change-point={e['cp']})"
    for e in R["reg_examples"]
)
skipped_total = sum(len(v.get("runs_skipped_fetch_error", [])) for v in meta["repos"].values())
low_yaml = [f"`{r}` ({v:.0f}%)" for r, v in per_repo.jobs_matched_to_yaml_pct.items() if v < 90]
junit_steps = int(per_repo.loc["junit-team/junit5", "steps"]) if "junit-team/junit5" in per_repo.index else None

ADOPT_WORD = {v: ("Adopted" if R["adoption"][v]["adopt"] else "Not adopted") for v in R["adoption"]}
_g = R["adoption"]["jeffreys_gate"]
_conf = P("pull_request", "original", "confirmed_flaky")
if _g["adopt"]:
    GATE_SUMMARY = "It passed the adoption rule and is now the default."
    GATE_DETAIL = "**Adopted.**"
else:
    GATE_SUMMARY = ("By the letter of the adoption rule it is **not adopted**, so the analyzer is unchanged; "
                    "it is the leading candidate once hand labels exist.")
    GATE_DETAIL = (
        f"**Not adopted.** It misses the recall limit by {_g['recall_drop'] - 0.20:.3f}, a margin smaller than one "
        f"confirmed job (1/{_conf} = {1 / _conf:.3f}). The rule is applied as written rather than relaxed after the fact. "
        "The analyzer keeps its original behaviour, and this gate is the first change to revisit once the hand labels "
        "measure what the lost recall is worth."
    )

report = f"""# ADPO on real CI data: evaluation report

Generated by `research/evaluation.ipynb` from the **{R['source']}** dataset. Every number below is
computed by the notebook; regenerate with the command in `research/README.md`.

**Summary**

- On real history, ADPO's original *flaky-job* rule was the weakest part of the engine. On
  pull-request runs only {fp(P('pull_request', 'original', 'precision_lb') * 100, 0)}% of the jobs it called flaky were confirmed by
  a re-run; on push-to-default-branch runs {P('push_default', 'original', 'tp')} of its {P('push_default', 'original', 'flagged')} flags were.
  {co['pct_cofail_ge_half']}% of the failures behind those flags were shared with at least half of the run's other jobs
  (co-failures), which points at broken commits and PRs rather than flaky jobs. **ADPO was wrong here**; see section e.
- A small-sample gate (>= {R['N_MIN']} runs and a one-sided 95% Jeffreys lower bound >= 5%) raised
  PR-run precision from {fp(P('pull_request', 'original', 'precision_lb'), 3)} to {fp(P('pull_request', 'jeffreys_gate', 'precision_lb'), 3)}, but recall fell by
  {fp(R['adoption']['jeffreys_gate']['recall_drop'], 3)} against a pre-stated limit of 0.20, and it cut push-run flags from
  {P('push_default', 'original', 'flagged')} to {P('push_default', 'jeffreys_gate', 'flagged')}.
  {GATE_SUMMARY}
  Excluding co-failures {'passed' if R['adoption']['drop_cofail']['adopt'] else 'did **not** pass'} the same rule.
- Job durations are skewed ({R['skew']['pct_skew_gt_1']}% of series have skewness > 1), which supports medians as the
  central statistic. For *tail* thresholds, {worst_tail} had the most series over 10% out-of-sample exceedance
  of the three estimators tested, and the CV-based branches in three analyzers are sensitive to single outliers.
- Without workflow-YAML `needs` edges, the parallelization analyzer would report {R['par_without_yaml']} candidate pairs;
  with them, {R['par_with_yaml']}. The backend still syncs no `needs`, so it is in the high-noise regime.
- The regression rule is conservative: its {reg['adpo']} flags are a subset of the {reg['mwu']} series a
  Mann-Whitney test flags.
- Precision of findings in general is **not yet measured**: 40 sampled findings await hand labels (section f).

## a. Data

Collected with `collect.py` (`collection_meta.json`: `collected_at` = {R['collected_at']} UTC; repos fetched before an
interrupted attempt were served from the response cache) through ADPO's own fetch layer from the public REST API: the **150 most recent completed runs** of one
CI workflow per repository ({R['n_repos']} repositories, listed with reasons in `repos.yaml`).
{T['runs']} runs ({T['attempts']} attempts including re-runs), {T['jobs']:,} jobs, {T['steps']:,} steps.
**Runs skipped because their jobs could not be fetched: {skipped_total}.**

{R['per_repo_table']}

Notes on the table:
- Repos with fewer than 150 runs are not truncated: GitHub listed 150 completed runs for each, but runs
  that never started a job (`action_required` fork PRs awaiting approval, and workflows that failed
  at startup) have no jobs and contribute nothing.
- Collection date ranges differ widely: busy repos cover days, quiet repos cover months.
- Job → workflow-YAML match rate overall: {R['yaml_match_pct_overall']}%. Below 90%: {', '.join(low_yaml) or 'none'}. Unmatched jobs get no `needs`.
- `junit-team/junit5` has only {junit_steps} step records for its jobs; step-level analysis is effectively empty for it.

Outcomes and handling:
- Job conclusions (%): {R['job_conclusion_pct']}. Final-attempt run conclusions (%): {R['run_conclusion_pct']}.
- Triggers (% of runs): {R['event_pct']}.
- Missing data: job timing missing {R['missing']['job_timing_missing_pct']}%, step timing missing {R['missing']['step_timing_missing_pct']}%.
  {R['missing']['job_negative_durations']} jobs have `completed_at` before `started_at`, all of them
  {R['missing']['negative_by_conclusion']} (median {R['missing']['negative_median_s']:.0f} s, i.e. timestamp rounding on jobs that never ran);
  ADPO's model treats negative durations as missing, and none are successful jobs. Runs with re-run attempts: {R['retried_runs']}.
- Duration statistics use **successful** executions only. Cancelled and skipped jobs are excluded from
  durations (a cancelled job's duration measures when it was killed, not how long it takes). Failure
  statistics use success/failure outcomes only. Outliers are **not** removed: robustness is the
  statistic's job, and removing them would hide exactly the variance some analyzers reason about.

## b. Distribution analysis

A *series* is the successful executions of one job in one repo, in time order; {R['n_series_20']} series have >= 20 runs.

![skew](figures/b1_skew.png)

- Median per-series skewness {R['skew']['median_skew']} (log scale: {R['skew']['median_log_skew']}); {R['skew']['pct_skew_gt_1']}% of series have
  skewness > 1 and Shapiro-Wilk rejects normality for {R['skew']['pct_shapiro_reject_normal']}%. The mean exceeds the median by more than 10%
  in {R['skew']['pct_mean_gt_median_10pct']}% of series. This **supports** ADPO's use of medians for "typical duration" and
  window comparisons, and argues against mean-based rules.
- **Tail thresholds** (all three target the same 2.3% one-sided tail; fit on the first 2/3 of each
  series, scored on the last 1/3):

{R['thr_table']}

  ![calibration](figures/b2_threshold_calibration.png)

  At the median series, **{closest}** is closest to the nominal 2.3% ({thr[closest]['median_exceed_pct']}%); the other two exceed
  nothing in most series, so they over-cover a typical job. Averaged over series, **{closest_mean}** is closest
  ({thr[closest_mean]['mean_exceed_pct']}%). **{worst_tail}** fails worst in the tail: {thr[worst_tail]['pct_series_exceed_over_10pct']}% of series exceed 10% out-of-sample,
  with a mean exceedance of {thr[worst_tail]['mean_exceed_pct']}%. MAD measures the width of the bulk, so median + k·MAD is well
  calibrated for a well-behaved series. It breaks on series with heavy tails or level shifts, which
  are exactly the series where a slow-run cut-off matters. That **challenges** the common advice to use it for
  slow-run cut-offs. No estimator is uniformly best; the empirical percentile needs the most data (largest
  bootstrap spread). ADPO itself has no per-run tail threshold; this matters if one is added.
- **CV vs robust CV.** Three analyzers branch on CV = SD/mean (savings `unknown` below 0.10,
  "high variance" wording at 0.25, dependency-install "cache possibly ineffective" at 0.30). At 0.30,
  CV and its robust analogue 1.4826·MAD/median disagree on {R['cv']['0.3']['disagree']}% of series ({R['cv']['0.3']['cv_only']} series above
  only by CV). Adding a single run at 3× the median moves CV by a median of {R['cv_one_outlier_median_shift']['cv']} and the robust
  CV by {R['cv_one_outlier_median_shift']['robust']}. So these branches can be flipped by one slow run. There is no ground truth for
  "the cache is ineffective", so this report flags the issue rather than changing the threshold.

  ![cv](figures/b3_cv_vs_robust.png)

## c. Sensitivity analysis

Findings across all {R['n_repos']} repos as each analyzer's main threshold varies (red line = default).

![sensitivity](figures/c1_sensitivity.png)

- `slow_job` / `slow_step` produce many findings at default settings ({R['small_n']['slow_job_defaults']} and {R['small_n']['slow_step_defaults']}). The count falls
  smoothly with the share threshold: there is no elbow that singles out a better default, so the defaults stay.
  {R['small_n']['slow_job_n_lt_5']} slow-job and {R['small_n']['slow_step_n_lt_5']} slow-step findings rest on fewer than 5 runs
  (`min_sample_size` is 1); they carry LOW confidence, but a minimum sample would be a reasonable follow-up once labels exist.
- `historical_regression`: {sens['historical_regression: min_pct_increase'][0.25]} findings at the 25% default, {sens['historical_regression: min_pct_increase'][0.1]} at 10%,
  {sens['historical_regression: min_pct_increase'][0.5]} at 50%.
- `dependency_install` is nearly flat ({min(sens['dependency_install: cache CV threshold'].values())}–{max(sens['dependency_install: cache CV threshold'].values())}): almost all of its findings are
  "no cache detected", not CV-driven.
- `failure_retry`: {sens['failure_retry: flaky_min_rate'][0.05]} jobs at the 5% rate threshold. The count is steep
  around the default ({sens['failure_retry: flaky_min_rate'][0.02]} at 2%, {sens['failure_retry: flaky_min_rate'][0.1]} at 10%), so the rate threshold alone is a fragile
  control. Section e tests "rate credibly above a line" instead.
- `potential_parallelization`: {sens['parallelization: min_sequential_consistency'][0.75]} pairs at the 0.75 default. The bigger lever is data, not the
  threshold: {R['par_with_yaml']} pairs with YAML `needs` vs {R['par_without_yaml']} when `needs` is empty (what the backend currently stores).

## d. Regression detection

{R['reg_n_series']} job series with >= 12 successful runs. Flags: ADPO {reg['adpo']}, Mann-Whitney U (BH q < 0.05) {reg['mwu']},
MWU + ADPO's effect-size gates {reg['mwu_eff']}, change-point (PELT, L1 on log durations) {reg['cp']}.

{R['reg_confusion_adpo_mwu']}

Cohen's κ: {R['reg_kappa']}

- Every ADPO flag is also MWU-significant (ADPO-only: {R['reg_adpo_not_mwu']}). MWU flags {reg['mwu'] - reg['adpo']} more series, but
  MWU with ADPO's effect-size gates is identical to ADPO (κ = {R['reg_kappa'].get('adpo vs mwu_eff')}). The extra MWU flags are
  statistically detectable shifts smaller than +25% / +20 s. ADPO's rule behaves like "significant **and** practically large".
- Change-point and ADPO disagree more (ADPO-only {R['reg_adpo_not_cp']}, change-point-only {R['reg_cp_not_adpo']}). Change-point finds level shifts
  anywhere in the series and compares the last segment with the one before it, while ADPO compares fixed
  60/40 windows. A step change near the start of the recent window can satisfy one rule and not the other.
- There is no ground truth for "real regression", so none of these is shown to be better; the
  disagreements are the cases to label. ADPO's rule is unchanged.

{ex_lines}

![regression examples](figures/d1_regression_examples.png)

## e. Flakiness: rates, a re-run ground truth, and co-failures

**Minimum runs (N_MIN).** Rule used: **{R['n_min_rule']}** → **N_MIN = {R['N_MIN']}**. For each job with >= 2N runs,
two disjoint random sets of N runs were drawn (20 seeded repeats) and their failure rates correlated
across jobs. Split-half Spearman reaches 0.7 first at N = {R['N_MIN']} (max over the grid {R['max_split_half']:.3f}). Below that,
two samples of the same job disagree too much to rank on. The Jeffreys fallback was not needed (median observed failure rate {R['median_failure_rate']*100:.2f}%).
{R['jobs_below_n_min']} of {R['jobs_total']} jobs have fewer than N_MIN runs and get **no rate and no rank**.

{R['n_min_table']}

**Ranking before/after the Jeffreys posterior** (Beta(k+½, n−k+½), jobs with n >= N_MIN; overlap of the two top-10 lists: {R['fail_rank_overlap_top10']}).

By raw final-attempt failure rate:

{R['fail_top_raw']}

By one-sided 95% posterior lower bound:

{R['fail_top_post']}

![posterior](figures/e1_failure_posterior.png)

**Re-run ground truth.** A *flake event* is a run (one `head_sha`) where a job failed on an earlier attempt and
passed on a later one. Flake events: {R['flake_events_by_subset']}, from re-run runs {R['reruns_by_subset']} out of runs {R['runs_by_subset']}.
{R['flake_jobs_with_event']} jobs have at least one. Jobs nobody re-ran cannot be confirmed, so **precision is a lower bound**
(`precision_among_rerun_jobs` restricts to flagged jobs that were re-run at least once).

Top jobs by flake-event lower bound (n >= N_MIN):

{R['flake_top_post']}

**ADPO's flaky flags vs the ground truth** (Wilson 95% intervals; `original` = rule before this evaluation):

{R['flaky_pr_table']}

**Adoption rule.** The author stated it before this final table, but after seeing a preliminary version, so it was
not pre-registered. On PR runs, a variant needs precision up by >= 0.10 with non-overlapping Wilson intervals,
recall down by <= 0.20 (absolute), and no confirmed-flaky job lost on push-to-default runs. Outcome per variant:
{R['adoption']}

- `jeffreys_gate`, pull requests: precision {fp(P('pull_request','original','precision_lb'),3)} → {fp(P('pull_request','jeffreys_gate','precision_lb'),3)}
  (intervals separated: {R['adoption']['jeffreys_gate']['ci_separated']}), recall {fp(P('pull_request','original','recall'),3)} → {fp(P('pull_request','jeffreys_gate','recall'),3)}
  (drop {fp(R['adoption']['jeffreys_gate']['recall_drop'],3)}). Push to default: flags {P('push_default','original','flagged')} → {P('push_default','jeffreys_gate','flagged')}, with
  {P('push_default','original','tp')} confirmed among the originals. {GATE_DETAIL}
  An earlier version of this table, which did not yet mirror the analyzer's final-attempt semantics, put the drop at
  0.192. The gate was briefly committed on that basis and then reverted (commits 32ac916 and 88ce5e8).
- `drop_cofail`, pull requests: precision {fp(P('pull_request','drop_cofail','precision_lb'),3)}, recall {fp(P('pull_request','drop_cofail','recall'),3)}.
  **{ADOPT_WORD['drop_cofail']}**: the precision gain is within the noise and recall drops by more than half.
- `both`: recall {fp(P('pull_request','both','recall'),3)}. **{ADOPT_WORD['both']}.**

**Co-failure: a case where ADPO was wrong.** Under the original rule, ADPO flagged {co['flagged_jobs']} (subset, job) pairs as
flaky across {co['repos_with_flagged_jobs']} repos. Of their {co['failing_executions_of_flagged_jobs']:,} failing final-attempt executions,
**{co['pct_cofail_ge_half']}%** happened in runs where at least half of the *other* jobs also failed
({co['pct_cofail_by_subset']} by subset). For {co['flagged_jobs_majority_cofail']} flagged pairs, most failures were co-failures,
and in {co['repos_majority_cofail']} of {co['repos_with_flagged_jobs']} repos most flagged failures were. Job failure rate by trigger (%): {R['fail_rate_by_subset_pct']}.
For those jobs, "flaky, look for non-determinism" was the wrong diagnosis in most cases: the run failed
because the commit or the environment broke, and most jobs reported it. On push-to-default runs, none of the
original flags was confirmed by a re-run.

The obvious fix, dropping co-failed runs, does not survive the ground truth: confirmed-flaky jobs co-fail too (runner
or network problems hit several jobs at once), so it costs most of the recall. Re-runs are the better
ground truth because they hold the commit fixed. A failure that disappears on retry of the same SHA is evidence
about the job; a failure shared by every job is evidence about the commit.

![cofailure](figures/e2_cofailure_by_repo.png)

## f. Precision check (hand labels)

All findings at the shipped defaults: {R['findings_default_total']} ({R['findings_default']}).
40 were sampled with seed {SEED}, stratified so every analyzer is represented ({R['label_alloc']}), into `labels.csv`.
The full finding JSON for each is in `data/sampled_findings.json`. Precision per analyzer with Wilson 95% intervals:

{R['precision_table']}

## g. Limitations and threats to validity

- **Repo selection.** Popular, well-maintained projects picked by hand for ecosystem spread, not sampled.
  Their CI is probably healthier than average, so ADPO's findings rate here likely understates a typical repo's.
  One workflow per repo; release, docs and scheduled workflows are not represented.
- **Time window.** 150 runs covers two days for some repos and a year for others. Regression and flakiness
  estimates are about different time scales across repos, and short windows can't show slow drift.
- **Runner variance.** Hosted runners vary in hardware and load, so duration changes may reflect GitHub's fleet, not the
  repo. No runner metadata beyond labels is available to control for it.
- **Mixed triggers.** PR runs test unmerged code and fail for reasons unrelated to CI health. Section e separates
  them; sections b–d pool them.
- **Ground truth.** Re-runs are chosen by humans (a maintainer retries what they *believe* is flaky), so the flake
  truth is biased toward jobs people already suspect, and it misses flaky jobs nobody re-ran. Precision is a lower
  bound; recall is relative to *noticed* flakiness.
- **Small samples.** {R['jobs_below_n_min']} jobs fall below N_MIN. Several repos contribute few re-runs, so the push-to-default
  ground truth has only {P('push_default','original','confirmed_flaky')} confirmed jobs. Its recall is not estimable with useful precision.
- **Researcher degrees of freedom.** N_MIN's 0.7 reliability level, the 50% co-failure line, the change-point
  penalty and the adoption threshold were chosen by the author during analysis. They are stated so they can be
  challenged, and the notebook makes each one a single-line change.
- **No labels yet.** Until `labels.csv` is filled in, nothing here measures whether findings are *useful*.
"""
(HERE / "REPORT.md").write_text(report, encoding="utf-8")
(HERE / "data" / "results.json").write_text(json.dumps(R, indent=1, default=str) + "\n")
print(len(report), "chars written")